In [33]:
from experiment_management import (
    BatchSteer,
    VectorRepository,
    ExperimentLiveInstanceRepository,
    ExperimentSnapshotRepository,
    ExperimentTemplateRepository,
    GeneratedOutputRepository,
    MetricRepository,
    LiveInstance,
    PromptDTO,
    PromptRepository,
    init_schema,
    register_loss_fn,
)
import json
import torch
import math

In [34]:
#initialise database and persistance
init_schema()
prompt_repo = PromptRepository()
et_repo = ExperimentTemplateRepository()
vec_repo = VectorRepository()
live_repo = ExperimentLiveInstanceRepository(vector_repo=vec_repo)
snap_repo = ExperimentSnapshotRepository(vector_repo=vec_repo)
output_repo = GeneratedOutputRepository()
metric_repo = MetricRepository()

In [35]:
prompts = ['Write a reciepe for tomato soup',
]
prompt_chat_format = [[
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
        ],
    },
] for prompt in prompts]


group = prompt_repo.create_group_from_strings(json.dumps(prompt_chat) for prompt_chat in prompt_chat_format)
print(group)
group_id = group.group_id

PromptGroupDTO(group_id=1)


In [36]:
#create loss function and register it in application before liveinstances are initialised
def melbo_lesswrong_loss(steered_output, vanilla_output, input_ids, p: int, q: int, target_layer: int):
    steered_hidden = steered_output.hidden_states[target_layer]
    vanilla_hidden = vanilla_output.hidden_states[target_layer]
    delta = steered_hidden - vanilla_hidden
    token_norm = delta.norm(dim=-1).pow(p).sum(dim=1)
    loss = -token_norm.pow(1 / q).sum()
    return loss
register_loss_fn("melbo_lesswrong_loss", melbo_lesswrong_loss)

In [37]:
loss_kwargs = {"p": 2, "q": 2, "target_layer": -1}
loss_kwargs_json = json.dumps(loss_kwargs)
optimizer_kwargs = {"lr": 1e-3}
optimizer_kwargs_json = json.dumps(optimizer_kwargs)


et_dto = et_repo.create_from_args(
    prompt_group=group_id,
    loss_name="melbo_lesswrong_loss",
    loss_additional_parameters=loss_kwargs_json,
    optimizer_name="AdamW",
    optimizer_additional_parameters=optimizer_kwargs_json,
    module_name="model.language_model.layers.6",
    batch_size=1,
    normalization=1.0,
)
print("et_dto: "+ str(et_dto.experiment_template_id))

et_dto: 1


In [38]:
# #model loading
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend
model_id = "DominicJW/Ministral-3-3B-Instruct-2512-BF16-bnb-4bit"
model_id = "/home/u/.cache/huggingface/hub/models--DominicJW--Ministral-3-3B-Instruct-2512-BF16-bnb-4bit/snapshots/43aec86bebdbb2a86e6c5878123a0fdf29e48612"
tokenizer = MistralCommonBackend.from_pretrained(model_id, local_files_only=True)
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id, device_map="cuda", local_files_only=True
)
for param in model.parameters():
    param.requires_grad = False

vector_width = model.config.text_config.hidden_size
vector_shape = (vector_width,)

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

In [39]:
et_dto_1 = et_repo.get(et_dto.experiment_template_id) 
et_dto_1.normalization = 15.0
et_dto_1 = et_repo._create(et_dto_1) #creates new dto and db entry with updated value


In [41]:
seeds = [58203,14324,95232,85312,45925] #starting with the same vectors so we can see how normalization changes it
vec_dtos = [vec_repo.create_from_seed(seed=seed,shape=vector_shape) for seed in seeds]
live_dtos = [live_repo.create_from_vec_dto(experiment_template_dto=et_dto_1,vec_dto=vec_dto) for vec_dto in vec_dtos]

live_objs = [LiveInstance(live_dto,live_repo,et_repo,snap_repo) for live_dto in live_dtos]
batch_steer = BatchSteer(live_objs, model)

prompt_dtos = prompt_repo.get_prompts_from_group(group_id)
batch_prompts = [json.loads(prompt_dto.text) for prompt_dto in prompt_dtos]
vanilla_inputs = tokenizer.apply_chat_template(batch_prompts, return_tensors="pt", padding=True).to(model.device)
vanilla_input_ids = vanilla_inputs.input_ids
vanilla_attention_mask = vanilla_inputs.attention_mask

steered_input_ids = vanilla_input_ids.repeat(len(batch_steer.live_objs), 1)
steered_attention_mask = vanilla_attention_mask.repeat(len(batch_steer.live_objs), 1)

with torch.no_grad(), batch_steer.steer():
    generated_ids = model.generate(
        input_ids=steered_input_ids,
        attention_mask=steered_attention_mask,
        max_new_tokens=100,
    )[:,vanilla_input_ids.shape[1]:]
    generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
#proccessing generated text
for i, text in enumerate(generated_text):
    prompt_id = prompt_dtos[i % len(prompt_dtos)].prompt_id
    live_index = i // len(prompt_dtos)
    live_obj = live_objs[live_index]
    #saving responses to repo
    steered_output = output_repo.create_from_snapshot(
        live_obj.create_snapshot(save_vector=False),
        text,
        prompt_id=prompt_id,
        vanilla=False,
    )

    #printing responses out
    screen_width = 200
    if (i % len(prompt_dtos)) == 0:
        title = f"Initial Vector: {live_obj.live_dto.initial_vector_id} Normalization: {live_obj.et_dto.normalization}" 
        t = screen_width-len(title)
        space_lhs = math.floor(t/2)
        space_rhs = screen_width - (space_lhs+len(title))
        print()
        print(" " + "-"*screen_width)
        print("#" + (" "*(screen_width)) + "#")
        print("#" + (" "* space_lhs) + title + (" "* space_rhs) + "#")
        print("#" + (" "*(screen_width)) + "#")
        print(" " + "-"*screen_width)
    else:
        print(" " + "-"*screen_width)
    title = f"Prompt: {prompt_id}"
    t = screen_width-len(title)
    space_lhs = math.floor(t/2)
    space_rhs = screen_width - (space_lhs+len(title))
    print("#" + (" "* space_lhs) + title + (" "* space_rhs) + "#")
    print(" " + "-"*screen_width)
    
    print()
    print(text)


 --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#                                                                                                                                                                                                        #
#                                                                                 Initial Vector: 1 Normalization: 15.0                                                                                  #
#                                                                                                                                                                                                        #
 -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
generated_ids = model.generate(
    input_ids=vanilla_input_ids,
    attention_mask=vanilla_attention_mask,
    max_new_tokens=40,
)[:,vanilla_input_ids.shape[1]:]
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)

indent_unit = "    "
print_wrapped("Vanilla generated text:", "")
for i, text in enumerate(generated_text):
    prompt_id = prompt_dtos[i % len(prompt_dtos)].prompt_id
    vanilla_out = output_repo.create_vanilla(
        text=text,
        prompt_id=prompt_id
    )
    print_wrapped(f"prompt[{i}] id={prompt_id} output_id={vanilla_out.output_id}", indent_unit)
    print_wrapped(text, indent_unit)


In [13]:
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
print("Vanilla generated text: ")
for i, text in enumerate(generated_text):
    prompt_id = prompt_dtos[i % len(prompt_dtos)].prompt_id
    vanilla_out = output_repo.create_vanilla(
        text=text,
        prompt_id=prompt_id
    )
    print("prompt: "+ str(prompt_id) + " output_id: " + str(vanilla_out.output_id))
    print(text)

Vanilla generated text: 
prompt: 1 output_id: 7
Here’s a simple, comforting **Classic Tomato Soup Recipe** that’s creamy, flavorful, and perfect for a cozy meal. You can serve it with crusty bread, gr


In [12]:
#training loop
vanilla_output = model(**vanilla_inputs, output_hidden_states=True)
for step in range(1, 101):
    with batch_steer.steer():
        steered_output = model(
            input_ids=steered_input_ids,
            attention_mask=steered_attention_mask,
            output_hidden_states=True,
        )
    loss = batch_steer.calc_loss(steered_output, vanilla_output, vanilla_inputs.input_ids)
    loss.backward()
    if step % 5 == 0:
        save_vectors_flag = step % 10 == 0
        save_generated_text_flag = save_vectors_flag #can change
        _ = [live_obj.create_snapshot(save_vector=save_vectors_flag) for live_obj in live_objs]
        for live_obj in live_objs:
            if live_obj.last_training_loss is not None:
                metric = metric_repo.create_from_snapshot(
                    live_obj.last_snapshot,
                    value=float(live_obj.last_training_loss.detach().item()),
                    description="Training Loss",
                )
        if save_vectors_flag:
            with batch_steer.steer():                
                generated_ids = model.generate(
                    input_ids=steered_input_ids,
                    attention_mask=steered_attention_mask,
                    max_new_tokens=40,
                )[:,vanilla_input_ids.shape[1]:]
                generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
                for i, text in enumerate(generated_text):
                    live_idx = i // len(prompt_dtos)
                    steered_output = output_repo.create_from_snapshot(
                        live_objs[live_idx].last_snapshot,
                        text,
                        prompt_id=prompt_dtos[i % len(prompt_dtos)].prompt_id,
                        vanilla=False,
                    )
    for live_obj in live_objs:
        live_obj.step_optimizer()
        live_obj.iteration_count += 1
        live_obj.update_repo()